## **Building a Simple Workflow with Chains**

Now let's build a simple workflow that chains together different LLM calls. Remember from Week 1, we discussed how workflows differ from agents - workflows follow a predetermined sequence of steps with clear inputs and outputs, while agents can make decisions about which actions to take.

In this section, we'll implement a research assistant workflow that:
1. Summarizes a topic
2. Identifies key questions
3. Answers those questions

Instead of making a single LLM call to handle all these tasks at once (which could lead to less focused results), we're breaking it into three distinct LLM calls and chaining them together. The output of each step becomes the input to the next, creating a more structured and reliable process.

## Import the necessary libraries

In [8]:
from langchain.chains import SequentialChain  # Allows chaining multiple LLM operations in sequence
from langchain.chains.llm import LLMChain  # Base class for creating single-step LLM operations
from langchain_core.prompts import PromptTemplate  # Template for creating structured prompts with variables
from langchain_core.output_parsers import CommaSeparatedListOutputParser  # Parser that converts LLM output into a comma-separated list
from langchain_groq import ChatGroq # For using Groq LLM

## Initialize our LLM

In [9]:
llm = ChatGroq(
    model="llama-3.1-8b-instant",
    temperature=0.7,
    api_key="put your groq api key here",
)

## Below we will create a workflow that chains together different LLM calls.

In [10]:
# Step 1: Create a summarization chain
summarize_prompt = PromptTemplate(
    input_variables=["topic"],
    template="Provide a concise summary of the following topic: {topic}"
)
summarize_chain = summarize_prompt | llm

# Step 2: Create a questions generation chain
question_prompt = PromptTemplate(
    input_variables=["summary"],
    template="Based on this summary: {summary}\n\nGenerate 3 insightful questions that would help someone understand this topic better. Return them as a comma-separated list."
)
question_chain = question_prompt | llm

# Step 3: Create an answer chain
answer_prompt = PromptTemplate(
    input_variables=["summary", "questions"],
    template="Based on this summary: {summary}\n\nAnd these questions: {questions}\n\nProvide a detailed answer to the first question."
)
answer_chain = answer_prompt | llm

## Connect the chains into a complete workflow

In [11]:
def research_workflow(topic):
    # Run summarize chain
    summary_msg = summarize_chain.invoke({"topic": topic})
    summary = summary_msg.content
    
    # Run question chain with the summary
    questions_msg = question_chain.invoke({"summary": summary})
    questions = questions_msg.content
    
    # Run answer chain with both summary and questions
    answer_msg = answer_chain.invoke({"summary": summary, "questions": questions})
    answer = answer_msg.content
    
    # Return all results
    return {
        "summary": summary,
        "questions": questions,
        "answer": answer
    }

In [12]:
# Run the entire workflow
results = research_workflow("Quantum Computing")

print("SUMMARY:")
print(results["summary"])
print("\nQUESTIONS:")
print(results["questions"])
print("\nANSWER TO FIRST QUESTION:")
print(results["answer"])

SUMMARY:
**Quantum Computing: A Concise Summary**

**What is Quantum Computing?**
Quantum computing is a new paradigm of computing that uses the principles of quantum mechanics to perform calculations and operations on data. It leverages the unique properties of quantum systems, such as superposition, entanglement, and interference, to solve complex problems that are beyond the capabilities of classical computers.

**Key Principles:**

1. **Quantum bits (qubits)**: The fundamental unit of quantum information, which can exist in multiple states simultaneously.
2. **Superposition**: A qubit can represent multiple values at the same time, allowing for parallel processing.
3. **Entanglement**: Qubits can be connected in a way that allows them to affect each other, even at a distance.
4. **Quantum gates**: The quantum equivalent of logic gates, used to perform operations on qubits.

**Advantages:**

1. **Exponential scaling**: Quantum computers can solve certain problems much faster than cl

This workflow approach offers several advantages:

1. Specialized prompts: Each step has a focused prompt optimized for its specific task
2. Decomposition: Breaking complex tasks into smaller steps makes the overall process more reliable
3. Transparency: You can inspect the intermediate outputs at each stage
4. Flexibility: You can modify individual steps without rewriting the entire system
5. Reusability: Components can be reused in different workflows

This pattern of connecting specialized LLM calls is fundamental to building robust AI applications and will be crucial as we move toward more complex agent architectures in future weeks.

In all the examples, we used a very basic system prompt, but remember to always make the system prompt as detailed and specific as possible, because it sets the behavior, tone, and role of the AI throughout the interaction. A well-crafted system prompt helps guide the model more effectively, ensuring it responds in a way that aligns with your goals, whether that’s being concise, technical, friendly, or anything else your use case requires.
